In [ ]:
# !pip install -r requirements.txt
# !pip install tensorboard
# !pip install wandb

In [1]:
import os
import numpy as np
import pandas as pd
import pickle as pkl

from model import Model
from data_handler import DataHandler

In [2]:
data_dir = 'Data'
os.makedirs(data_dir, exist_ok=True)
ticker = 'AAPL'
start_date = '2024-07-01'
end_date = '2024-08-30'
data_handler = DataHandler(data_dir)

In [3]:
processed_data = data_handler.get_data(ticker, start_date, end_date, split_train_test=True)
processed_train_data, processed_test_data = processed_data

File Data/AAPL_2024-07-01_2024-08-30.csv already exists, skipping download


In [4]:
model_dir = 'Model'
# os.makedirs(model_dir, exist_ok=True)
logging_dir = 'logs'

# trial = "_env6_trial1_totpen_multi_env_exp_rew_price_sde_false_allinv_buy_100xrewardis_extraMLP"
trial = "_env6_sell"
n_env = 8

training_params = {
    "total_timesteps": 5e5,
    "callback" : None
}

training_config = None

env_params = {
    "preferred_timeframe" : 390,
    "inventory": 10000,
    "action" : "sell",
}

model_name = f"{ticker}_trial{trial}.pt"

policy = "MlpPolicy"
config = {
    "experiment_name": f"{ticker}_trial{trial}",
    "policy_type": "MlpPolicy",
    "total_timesteps": training_params["total_timesteps"],
    "trial_name": trial,
    "log_dir" : logging_dir,
    "env_details" : env_params,
}
sac_model = Model(model_dir=model_dir, logging_dir=logging_dir,n_env=n_env, wandb_logs=False)

In [ ]:
import wandb

wandb.login(relogin=True, force=True)
run = wandb.init(
    project="Paper-replication",
    config=config,
    sync_tensorboard=True,  # auto-upload sb3's tensorboard metrics
)
sac_model = Model(model_dir=model_dir, logging_dir=logging_dir,n_env=n_env, wandb_logs=True)
training_params['tb_log_name'] =f'{run.id}'
trained_model = sac_model.train(processed_train_data, resume=False, model_name=model_name, env_params=env_params, training_config=training_config, training_params=training_params)

In [6]:
env_params["preferred_timeframe"] = 390
rew, trades = sac_model.test(processed_test_data[1000:], model=None, model_name=model_name, env_params=env_params)
print(" Reward: ", rew)
trades.to_csv(f"{ticker}_trades_sell.csv")
trades

/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/gym/spaces/box.py:127: UserWarning: WARN: Box bound precision lowered by casting to float32
  logger.warn(f"Box bound precision lowered by casting to {self.dtype}")
/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/stable_baselines3/common/vec_env/patch_gym.py:49: UserWarning: You provided an OpenAI Gym environment. We strongly recommend transitioning to Gymnasium environments. Stable-Baselines3 is automatically wrapping your environments in a compatibility layer, which could potentially cause issues.
  warnings.warn(
/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/gym/spaces/box.py:127: UserWarning: WARN: Box bound precision lowered by casting to float32
  logger.warn(f"Box bound precision lowered by casting to {self.dtype}")
/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/stable_baselines3/common/vec_env/patch_gym.py:49: UserWarning: You provid

Using cuda device
Model loaded from Model/AAPL_trial_env6_sell.pt
 Reward:  8145.818779083339


/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/gym/spaces/box.py:127: UserWarning: WARN: Box bound precision lowered by casting to float32
  logger.warn(f"Box bound precision lowered by casting to {self.dtype}")
/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/stable_baselines3/common/vec_env/patch_gym.py:49: UserWarning: You provided an OpenAI Gym environment. We strongly recommend transitioning to Gymnasium environments. Stable-Baselines3 is automatically wrapping your environments in a compatibility layer, which could potentially cause issues.
  warnings.warn(


,inventory,step,timestamp,time_elapsed,time_slice,shares,order_type,price,IS,IS_diff,IS_twap,best_price,best_twap_price,reward
0,9223.0,34,2024-08-22 19:25:00,34,34,777.0,market,224.64,2.069554e+06,2.069551e+06,2.252564,224.390000,226.000846,0.000000
1,8433.0,69,2024-08-23 13:30:00,69,35,790.0,market,224.39,1.899906e+06,1.899903e+06,2.252564,225.295000,226.000846,0.000000
2,7501.0,105,2024-08-23 14:06:00,105,36,932.0,market,226.20,1.694728e+06,1.694726e+06,2.252564,225.960000,226.000846,0.000000
3,6555.0,141,2024-08-23 14:42:00,141,36,946.0,market,227.29,1.483130e+06,1.483128e+06,2.252564,226.307500,226.000846,0.000000
4,5650.0,176,2024-08-23 15:17:00,176,35,905.0,market,227.35,1.278517e+06,1.278515e+06,2.252564,226.342000,226.000846,0.000000
5,4587.0,211,2024-08-23 15:52:00,211,35,1063.0,market,226.48,1.037697e+06,1.037695e+06,2.252564,226.281667,226.000846,0.000000
6,3677.0,244,2024-08-23 16:25:00,244,33,910.0,market,225.98,8.312459e+05,8.312437e+05,2.252564,226.134286,226.000846,0.000000
7,2138.0,254,2024-08-23 16:35:00,254,10,1539.0,market,225.25,4.834671e+05,4.834648e+05,2.252564,226.036250,226.000846,0.000000
8,1211.0,259,2024-08-23 16:40:00,259,5,927.0,market,225.35,2.737591e+05,2.737568e+05,2.252564,225.932222,226.000846,0.000000
9,158.0,264,2024-08-23 16:45:00,264,5,1053.0,market,225.10,3.591724e+04,3.591499e+04,2.252564,225.810000,226.000846,0.000000


In [ ]:
# # buy training:
# trial = "_env6_buy"

# env_params = {
#     "preferred_timeframe" : 390,
#     "inventory": 10000,
#     "action" : "buy",
# }

# model_name = f"{ticker}_trial{trial}.pt"

# policy = "MlpPolicy"
# config = {
#     "experiment_name": f"{ticker}_trial{trial}",
#     "policy_type": "MlpPolicy",
#     "total_timesteps": training_params["total_timesteps"],
#     "trial_name": trial,
#     "log_dir" : logging_dir,
#     "env_details" : env_params,
# }
# run = wandb.init(
#     project="Paper-replication",
#     config=config,
#     sync_tensorboard=True,  # auto-upload sb3's tensorboard metrics
# )
# sac_model = Model(model_dir=model_dir, logging_dir=logging_dir,n_env=n_env, wandb_logs=True)
# training_params['tb_log_name'] =f'{run.id}'
# trained_model = sac_model.train(processed_train_data, resume=False, model_name=model_name, env_params=env_params, training_config=training_config, training_params=training_params)

In [ ]:
# env_params["preferred_timeframe"] = 390
# rew, trades = sac_model.test(processed_test_data[1000:], model=None, model_name=model_name, env_params=env_params)
# print(" Reward: ", rew)
# trades.to_csv(f"{ticker}_trades_buy.csv")
# trades

In [ ]:
10000/390

# Fine Tuning

In [ ]:
# from ray import train, tune
# from ray.tune.schedulers import ASHAScheduler
# from ray.tune.search.hyperopt import HyperOptSearch

In [ ]:
# # from ray import tune, train
# # from ray.tune.schedulers import ASHAScheduler
# # from ray.tune.search.hyperopt import HyperOptSearch
# # import ray

# model_dir = os.path.abspath('./Models_tuning')
# # os.makedirs(model_dir, exist_ok=True)
# print(model_dir)
# # No logging
# lstm_ppo_model = Model(model_dir=model_dir, stats_window_size=1000, logging_dir=None)

# # Training Config
# """
# Key required for fine tuning
# learning_rate, n_steps, batch_size, gamma, clip_range, n_epochs, ent_coef, resume, total_timesteps, callback, tb_log_name, train_datatest_data
# """

# training_params = {
#     "total_timesteps": 5000,
#     "callback" : None
# }

# finetune_config = {
#     "learning_rate": tune.loguniform(1e-5, 1e-1),
#     "n_steps": tune.choice([32, 64, 128, 256, 512]),
#     "batch_size": tune.choice([64, 128, 256]),
#     "gamma": tune.uniform(0.9, 0.999),
#     "clip_range" : tune.uniform(0.1, 0.4),
#     "n_epochs": tune.choice([4, 6, 8]),
#     "vf_coef" : tune.loguniform(0.1, 0.8),

#     "ent_coef": tune.loguniform(0.0001, 0.1),
#     "resume" : False,
#     "total_timesteps": training_params['total_timesteps'],
#     "callback" : training_params['callback'],
#     "env_params": {
#         "preferred_timeframe" : 390,
#         "inventory": 10000
#     }
# }

# ray.init(num_gpus=1)

# scheduler = ASHAScheduler()

# hyperopt_search = HyperOptSearch()


# finetune_config["train_data"] = processed_train_data
# finetune_config["test_data"] = processed_test_data

# results_dir = os.path.abspath("./tune_results_tmp")

# analysis = tune.Tuner(
#     lstm_ppo_model.fine_tuning_model,
#     param_space=finetune_config,
#     run_config=train.RunConfig(
#         name=f"{ticker}",
#         storage_path=results_dir,
#     ),
#     tune_config=tune.TuneConfig(
#         search_alg=hyperopt_search,
#         scheduler=scheduler,
#         num_samples=1,
#         trial_dirname_creator=lambda trial: trial.trial_id,
#         metric="reward",
#         mode="max",),
# )

# results = analysis.fit()
# print(results)
# results.get_dataframe().to_csv(f"{results_dir}/tune_results.csv")
# best_results = results.get_best_result(metric="reward", mode="max")
# # best_config = analysis.get_best_config(metric="reward", mode="max")

# print("Best config:", best_results.config)
# best_config = best_results.config
# # Save best parameters
# # save_dir = "./best_ppo_params"
# # os.makedirs(save_dir, exist_ok=True)
# del best_config["train_data"]
# del best_config["test_data"]
# pkl.dump(best_config, open(os.path.join(results_dir, "best_ppo_params.pkl"), "wb"))
# with open(os.path.join(results_dir, "best_config.txt"), "w") as f:
#     for key, value in best_config.items():
#         f.write(f"{key}: {value}\n")
# ray.shutdown()
# # print("Best config: ", analysis.get_best_config(metric="mean_reward", mode="max"))
# # df = analysis.results_df
# # print(df.head())

In [ ]:
# !pip install -U stable-baselines3
# !pip install -U hyperopt